In [6]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

In [7]:
df = pd.read_csv('TED_Talk.csv')
print(df.head())

   talk__id                                         talk__name  \
0        66                        Do schools kill creativity?   
1      2405  This is what happens when you reply to spam email   
2      1569           Your body language may shape who you are   
3       848                   How great leaders inspire action   
4      1042                         The power of vulnerability   

                                   talk__description  view_count  \
0  Sir Ken Robinson makes an entertaining and pro...    65678748   
1  Suspicious emails: unclaimed insurance bonds, ...    59725446   
2  (NOTE: Some of the findings presented in this ...    57734063   
3  Simon Sinek has a simple but powerful model fo...    50494918   
4  Brené Brown studies human connection -- our ab...    48503432   

   comment_count  duration                                         transcript  \
0         4952.0      1164  Good morning. How are you?(Audience) Good.It's...   
1          288.0       588  A fe

In [8]:
data = df[['talk__name','talk__description','transcript']]

data.head()

,talk__name,talk__description,transcript
0,Do schools kill creativity?,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you?(Audience) Good.It's...
1,This is what happens when you reply to spam email,"Suspicious emails: unclaimed insurance bonds, ...","A few years ago, I got one of those spam email..."
2,Your body language may shape who you are,(NOTE: Some of the findings presented in this ...,So I want to start by offering you a free no-t...
3,How great leaders inspire action,Simon Sinek has a simple but powerful model fo...,How do you explain when things don't go as we ...
4,The power of vulnerability,Brené Brown studies human connection -- our ab...,"So, I'll start with this: a couple years ago, ..."


In [9]:
data['talk__description'] = data['talk__description'].fillna('')
data['transcript'] = data['transcript'].fillna('')

In [10]:
data['content'] = data['talk__description'] + " " + data['transcript']

data.head()

,talk__name,talk__description,transcript,content
0,Do schools kill creativity?,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you?(Audience) Good.It's...,Sir Ken Robinson makes an entertaining and pro...
1,This is what happens when you reply to spam email,"Suspicious emails: unclaimed insurance bonds, ...","A few years ago, I got one of those spam email...","Suspicious emails: unclaimed insurance bonds, ..."
2,Your body language may shape who you are,(NOTE: Some of the findings presented in this ...,So I want to start by offering you a free no-t...,(NOTE: Some of the findings presented in this ...
3,How great leaders inspire action,Simon Sinek has a simple but powerful model fo...,How do you explain when things don't go as we ...,Simon Sinek has a simple but powerful model fo...
4,The power of vulnerability,Brené Brown studies human connection -- our ab...,"So, I'll start with this: a couple years ago, ...",Brené Brown studies human connection -- our ab...


In [11]:
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

tfidf_matrix = tfidf.fit_transform(data['content'])

tfidf_matrix.shape

(4609, 5000)

In [12]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

cosine_sim

array([[1.        , 0.36490026, 0.23149389, ..., 0.01251755, 0.01891654,
        0.01004119],
       [0.36490026, 1.        , 0.15152369, ..., 0.01079362, 0.00241403,
        0.01070366],
       [0.23149389, 0.15152369, 1.        , ..., 0.04827542, 0.02761848,
        0.0128861 ],
       ...,
       [0.01251755, 0.01079362, 0.04827542, ..., 1.        , 0.01150448,
        0.0044875 ],
       [0.01891654, 0.00241403, 0.02761848, ..., 0.01150448, 1.        ,
        0.04239396],
       [0.01004119, 0.01070366, 0.0128861 , ..., 0.0044875 , 0.04239396,
        1.        ]])

In [13]:
indices = pd.Series(data.index, index=data['talk__name']).drop_duplicates()

indices.head()

talk__name
Do schools kill creativity?                          0
This is what happens when you reply to spam email    1
Your body language may shape who you are             2
How great leaders inspire action                     3
The power of vulnerability                           4
dtype: int64

In [14]:
def recommend_talks(title, cosine_sim=cosine_sim):

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:6]

    talk_indices = [i[0] for i in sim_scores]

    return data['talk__name'].iloc[talk_indices]

In [15]:
recommend_talks("Do schools kill creativity?")

102               Bring on the learning revolution!
90           How to escape education's death valley
2430                         A one-man world summit
162                                 Nerdcore comedy
531     How to run a company with (almost) no rules
Name: talk__name, dtype: object

In [16]:
def recommend_with_score(title):

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:6]

    for i in sim_scores:
        print(data['talk__name'][i[0]], " -> similarity:", round(i[1],3))

In [17]:
recommend_with_score("Do schools kill creativity?")

Bring on the learning revolution!  -> similarity: 0.507
How to escape education's death valley  -> similarity: 0.481
A one-man world summit  -> similarity: 0.423
Nerdcore comedy  -> similarity: 0.421
How to run a company with (almost) no rules  -> similarity: 0.387


In [19]:
import pickle
pickle.dump(data, open("ted_data.pkl","wb"))
pickle.dump(cosine_sim, open("similarity.pkl","wb"))
pickle.dump(indices, open("indices.pkl","wb"))